In [0]:
%pip install langchain-community

In [0]:
df = spark.sql("select * from bright_data_amazon_best_seller_products_reviews_products_dataset.datasets.amazon_best_seller_products")
df.display()
df.printSchema()


In [0]:
import json
import re
from typing import Dict, Any
from decimal import Decimal
from datetime import date, datetime

from pyspark.sql import SparkSession
from langchain_community.chat_models import ChatDatabricks

# SPARK SESSION
spark = SparkSession.builder.getOrCreate()


# CONFIGURATION
chat_model = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",
    max_tokens=600,
    temperature=0.2,
    api_key="dap1382e4407f2868e3cdaf75cd6efa67a50"
)

# FUNCTION: PYTHON JSON SERIALIZER
def json_safe(obj):
    """
    Converts non-JSON-serializable objects into safe formats.
    """
    if isinstance(obj, Decimal):
        return int(obj) if obj % 1 == 0 else float(obj)
    if isinstance(obj, (date, datetime)):
        return obj.isoformat()
    return str(obj)


def normalize_dict(d: Dict[str, Any]) -> Dict[str, Any]:
    return {k: json_safe(v) for k, v in d.items()}

def extract_json(raw_output: str) -> str:
    if raw_output is None:
        return ""

    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", cleaned)
    cleaned = re.sub(r"\n?```$", "", cleaned)

    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    return match.group(0) if match else cleaned

# FUNCTION: LLM PRODUCT ANALYSIS
def llm_product_analysis(product_context: Dict[str, Any]) -> Dict[str, Any]:
    prompt = f"""
You are a senior e-commerce product analyst.

Below is a JSON object describing an Amazon best-seller product.

Your tasks:
1. Summarize the product and its positioning.
2. Assess pricing and discount attractiveness.
3. Evaluate demand strength using rating, reviews count, and seller count.
4. Identify potential risks.
5. Suggest 3–6 actionable insights.
6. Return STRICT JSON only.

Product JSON:
{json.dumps(product_context, indent=2)}

Required JSON schema:

{{
  "product_summary": "string",
  "price_assessment": "competitive | premium | discounted | unclear",
  "demand_strength": "high | medium | low",
  "risk_flags": ["string"],
  "actionable_insights": ["string"]
}}
"""

    response = chat_model.invoke(prompt)
    raw_output = getattr(response, "content", None) or str(response)
    clean_output = extract_json(raw_output)

    try:
        return json.loads(clean_output)
    except Exception:
        return {
            "product_summary": "Invalid LLM response",
            "price_assessment": "unclear",
            "demand_strength": "low",
            "risk_flags": ["LLM returned invalid JSON"],
            "actionable_insights": []
        }

# READ AMAZON BEST SELLER TABLE
table_name = (
    "bright_data_amazon_best_seller_products_reviews_products_dataset."
    "datasets.amazon_best_seller_products"
)

df = spark.table(table_name)

# SAMPLE DATA (CONTROL COST)
sample_df = (
    df.select(
        "TIMESTAMP",
        "TITLE",
        "BRAND",
        "DESCRIPTION",
        "INITIAL_PRICE",
        "FINAL_PRICE",
        "CURRENCY",
        "AVAILABILITY",
        "REVIEWS_COUNT",
        "RATING",
        "NUMBER_OF_SELLERS",
        "CATEGORIES",
        "ASIN",
        "DISCOUNT",
        "DOMAIN"
    )
    .where("TITLE IS NOT NULL")
    .limit(10)
)

# DRIVER-SIDE LLM EXECUTION (SAFE)
results = []

for row in sample_df.collect():
    raw_dict = row.asDict(recursive=True)
    product_json = normalize_dict(raw_dict)

    analysis = llm_product_analysis(product_json)

    results.append({
        "ASIN": product_json.get("ASIN"),
        "BRAND": product_json.get("BRAND"),
        "TITLE": product_json.get("TITLE"),
        "analysis": analysis
    })

# OUTPUT
print(json.dumps(results, indent=2))
